In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV
)

from sklearn.pipeline import Pipeline

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier

from sklearn.svm import LinearSVC

from sklearn.metrics import accuracy_score

import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("../data/processed/final_dataset.csv")

X = df["clean_text"]

y = df["label"]

In [3]:
X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

In [4]:
pipelines = {

    "logistic_regression": Pipeline([
        ("tfidf", TfidfVectorizer()),
        ("classifier", LogisticRegression())
    ]),

    "random_forest": Pipeline([
        ("tfidf", TfidfVectorizer()),
        ("classifier", RandomForestClassifier())
    ]),

    "svm": Pipeline([
        ("tfidf", TfidfVectorizer()),
        ("classifier", LinearSVC())
    ])
}

In [13]:
param_grids = {

    "logistic_regression": {

        "tfidf__max_features":[3000,5000],

        "classifier__C":[0.1,1,10],

        "classifier__max_iter":[5000]

    },

    "random_forest": {

    "tfidf__max_features": [3000, 5000],

    "classifier__n_estimators": [100, 200],

    "classifier__max_depth": [20, None]

},

    "svm": {

        "tfidf__max_features":[3000,5000],

        "classifier__C":[0.1,1,10],

        "classifier__max_iter":[5000]

    }

}

In [14]:
best_models = {}

results = []

In [15]:
for name in pipelines:

    print("="*60)

    print(name)

    print("="*60)

    grid = GridSearchCV(

        estimator=pipelines[name],

        param_grid=param_grids[name],

        cv=5,

        scoring="accuracy",

        n_jobs=-1

    )

    grid.fit(

        X_train,

        y_train

    )

    best_models[name] = grid.best_estimator_

    prediction = grid.predict(X_test)

    accuracy = accuracy_score(

        y_test,

        prediction

    )

    results.append({

        "Model":name,

        "Accuracy":accuracy,

        "Best Parameters":grid.best_params_

    })

    print(

        "Best Accuracy:",

        accuracy

    )

    print()

    print(

        grid.best_params_

    )

logistic_regression
Best Accuracy: 0.9379760609357998

{'classifier__C': 10, 'classifier__max_iter': 5000, 'tfidf__max_features': 5000}
random_forest
Best Accuracy: 0.9107725788900979

{'classifier__max_depth': None, 'classifier__n_estimators': 200, 'tfidf__max_features': 3000}
svm
Best Accuracy: 0.9423286180631121

{'classifier__C': 1, 'classifier__max_iter': 5000, 'tfidf__max_features': 5000}


In [16]:
results = pd.DataFrame(results)

results.sort_values(

    by="Accuracy",

    ascending=False,

    inplace=True
)

results

,Model,Accuracy,Best Parameters
2,svm,0.942329,"{'classifier__C': 1, 'classifier__max_iter': 5..."
0,logistic_regression,0.937976,"{'classifier__C': 10, 'classifier__max_iter': ..."
1,random_forest,0.910773,"{'classifier__max_depth': None, 'classifier__n..."


In [17]:
for name, model in best_models.items():

    joblib.dump(

        model,

        "../models/"+name+"_tuned.pkl"

    )

In [18]:
results.to_csv(

    "../models/hyperparameter_results.csv",

    index=False

)

In [19]:
previous = pd.read_csv(

    "../models/model_evaluation.csv"
)

previous

comparison = previous.merge(

    results,

    on="Model",

    suffixes=("_Before","_After")
)

comparison

,Model,Accuracy_Before,Precision,Recall,F1 Score,Accuracy_After,Best Parameters
0,svm,0.946681,0.951542,0.941176,0.946331,0.942329,"{'classifier__C': 1, 'classifier__max_iter': 5..."
1,logistic_regression,0.928183,0.947608,0.906318,0.926503,0.937976,"{'classifier__C': 10, 'classifier__max_iter': ..."
2,random_forest,0.905332,0.920814,0.886710,0.903441,0.910773,"{'classifier__max_depth': None, 'classifier__n..."


In [20]:
print("="*60)

print("Hyperparameter Tuning Completed")

print("="*60)

print(results)

Hyperparameter Tuning Completed
                 Model  Accuracy  \
2                  svm  0.942329   
0  logistic_regression  0.937976   
1        random_forest  0.910773   

                                     Best Parameters  
2  {'classifier__C': 1, 'classifier__max_iter': 5...  
0  {'classifier__C': 10, 'classifier__max_iter': ...  
1  {'classifier__max_depth': None, 'classifier__n...  
